<a href="https://colab.research.google.com/github/mariaaapetrovskaya/compling_nlp_course/blob/main/03_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 1 (5 балла)

Имплементируйте алгоритм Леска (описание есть в семинаре) и оцените качество его работы на датасете `data/corpus_wsd_50k.txt`

В качестве метрики близости вы должны попробовать два подхода:

1) Jaccard score на множествах слов (определений и контекста)
2) Cosine distance на эмбедингах sentence_transformers

В качестве метрики используйте accuracy (% правильных ответов). Предсказывайте только многозначные слова в датасете

Контекст вы можете определить самостоятельно (окно вокруг целевого слова или все предложение). Также можете поэкспериментировать с предобработкой для обоих методов.

In [1]:
!wget -q https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/corpus_wsd_50k.txt.zip
!unzip -o corpus_wsd_50k.txt.zip

Archive:  corpus_wsd_50k.txt.zip
  inflating: corpus_wsd_50k.txt      
  inflating: __MACOSX/._corpus_wsd_50k.txt  


In [4]:
import re
import numpy as np
import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

STOP = set(stopwords.words('english'))
with open('corpus_wsd_50k.txt', encoding='utf-8') as f:
    blocks = f.read().split('\n\n')

dataset = []
for block in blocks:
    if not block.strip():
        continue
    sent = [line.split('\t') for line in block.strip().split('\n')]
    if all(len(row) == 3 for row in sent):
        dataset.append(sent)
print('предложений:', len(dataset))


items = []
for si, sent in enumerate(dataset):
    tokens = [row[2].lower() for row in sent]
    for ti, (sense, lemma, word) in enumerate(sent):
        if not sense:
            continue
        gold_name = sense.split('%')[0] if '%' in sense else sense
        items.append((si, ti, lemma.lower(), gold_name, tokens))
print('многозначных вхождений:', len(items))

lemma_to_synsets = {}
for _, _, lemma, _, _ in items:
    if lemma not in lemma_to_synsets:
        lemma_to_synsets[lemma] = wn.synsets(lemma)

all_synsets = set()
for ss_list in lemma_to_synsets.values():
    all_synsets.update(ss_list)
all_synsets = list(all_synsets)
print('уникальных синсетов:', len(all_synsets))

synset_texts = [ss.definition().lower() for ss in all_synsets]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model.max_seq_length = 128

synset_embs = model.encode(
    synset_texts, batch_size=128, normalize_embeddings=True,
    show_progress_bar=True,
)

unique_contexts = [' '.join([row[2].lower() for row in sent]) for sent in dataset]
context_embs = model.encode(
    unique_contexts, batch_size=128, normalize_embeddings=True,
    show_progress_bar=True,
)

ss_to_idx = {ss: i for i, ss in enumerate(all_synsets)}

def jaccard(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0
def clean(tokens):
    return [t for t in tokens if t.lower() not in STOP and t.isalpha()]
def evaluate_jaccard():
    total, correct = 0, 0
    for si, ti, lemma, gold_name, tokens in items:
        synsets = lemma_to_synsets.get(lemma, [])
        if not synsets:
            continue
        ctx = set(tokens)
        best_ss, best_score = None, -1
        for ss in synsets:
            def_tokens = re.findall(r'\w+', ss.definition().lower())
            s = jaccard(ctx, def_tokens)
            if s > best_score:
                best_score, best_ss = s, ss
        total += 1
        if best_ss and best_ss.name().split('.')[0] == gold_name:
            correct += 1
    return correct / total if total else 0.0

def evaluate_jaccard_clean():
    total, correct = 0, 0
    for si, ti, lemma, gold_name, tokens in items:
        synsets = lemma_to_synsets.get(lemma, [])
        if not synsets:
            continue
        ctx = set(clean(tokens))
        if not ctx:
            continue
        best_ss, best_score = None, -1
        for ss in synsets:
            def_tokens = clean(re.findall(r'\w+', ss.definition().lower()))
            if not def_tokens:
                continue
            s = jaccard(ctx, def_tokens)
            if s > best_score:
                best_score, best_ss = s, ss
        total += 1
        if best_ss and best_ss.name().split('.')[0] == gold_name:
            correct += 1
    return correct / total if total else 0.0

def evaluate_embedding():
    total, correct = 0, 0
    for si, ti, lemma, gold_name, tokens in items:
        synsets = lemma_to_synsets.get(lemma, [])
        if not synsets:
            continue
        ctx_vec = context_embs[si]
        idxs = [ss_to_idx[ss] for ss in synsets]
        sims = cosine_similarity([ctx_vec], synset_embs[idxs])[0]
        best_ss = synsets[int(np.argmax(sims))]
        total += 1
        if best_ss.name().split('.')[0] == gold_name:
            correct += 1
    return correct / total if total else 0.0

acc_j       = evaluate_jaccard()
acc_j_clean = evaluate_jaccard_clean()
acc_e       = evaluate_embedding()

print(f'Accuracy jaccard):              {acc_j:.4f}')
print(f'Accuracy jaccard и stopwords):  {acc_j_clean:.4f}')
print(f'Accuracy embedding):            {acc_e:.4f}')
print(f'Разница в jaccard:                 {acc_j_clean - acc_j:+.4f}')

предложений: 5801
многозначных вхождений: 53707
уникальных синсетов: 30636


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/240 [00:00<?, ?it/s]

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Accuracy jaccard):              0.6698
Accuracy jaccard и stopwords):  0.6846
Accuracy embedding):            0.6903
Разница в jaccard:                 +0.0148


# Задание 2 (5 балла)
Попробуйте разные алгоритмы кластеризации на датасете - `https://github.com/nlpub/russe-wsi-kit/blob/initial/data/main/wiki-wiki/train.csv`

Используйте код из семинара как основу. Используйте ARI как метрику качества.

Попробуйте все 4 алгоритма кластеризации, про которые говорилось на семинаре. Для каждого из алгоритмов попробуйте настраивать гиперпараметры (посмотрите их в документации). Прогоните как минимум 5 экспериментов (не обязательно успешных) с разными параметрами на каждый алгоритме кластеризации и оцените: качество кластеризации, скорость работы, интуитивность параметров.

Помимо этого также выберите 1 дополнительный алгоритм кластеризации отсюда - https://scikit-learn.org/stable/modules/clustering.html , опишите своими словами принцип его работы  и проделайте аналогичные эксперименты.

In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import (
    KMeans, DBSCAN, AffinityPropagation, AgglomerativeClustering,
    SpectralClustering
)
from sklearn.metrics import adjusted_rand_score
from sentence_transformers import SentenceTransformer
import time


url = ('https://raw.githubusercontent.com/nlpub/russe-wsi-kit/'
       'initial/data/main/wiki-wiki/train.csv')
df = pd.read_csv(url, sep='\t')
grouped = df.groupby('word')[['word', 'context', 'gold_sense_id']]


model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def embed_texts(texts):
    return model.encode(texts, normalize_embeddings=True,
                        show_progress_bar=False)
def run_experiment(cluster, X, y_true):
    t0 = time.perf_counter()
    labels = cluster.fit_predict(X)
    elapsed = time.perf_counter() - t0
    labels = labels + 1
    ari = adjusted_rand_score(y_true, labels)
    return ari, elapsed

def eval_algorithm(name, cluster_cls, params_list, X_all, y_all):
    print(f'\n{name}')
    for params in params_list:
        ari_list, time_list = [], []
        for key, grp in grouped:
            texts = grp['context'].values
            X = embed_texts(texts)
            y = grp['gold_sense_id'].values
            cluster = cluster_cls(**params)
            ari, el = run_experiment(cluster, X, y)
            ari_list.append(ari)
            time_list.append(el)
        print(f'  params={params}')
        print(f'    средний ARI: {np.mean(ari_list):.4f}  '
              f'суммарное время: {np.sum(time_list):.2f} c')


kmeans_params = [
    {'n_clusters': 2, 'n_init': 10, 'random_state': 0},
    {'n_clusters': 3, 'n_init': 10, 'random_state': 0},
    {'n_clusters': 4, 'n_init': 10, 'random_state': 0},
    {'n_clusters': 5, 'n_init': 10, 'random_state': 0},
    {'n_clusters': 6, 'n_init': 10, 'random_state': 0},
]
eval_algorithm('KMeans', KMeans, kmeans_params, None, None)

dbscan_params = [
    {'eps': 0.1, 'min_samples': 1},
    {'eps': 0.2, 'min_samples': 1},
    {'eps': 0.3, 'min_samples': 2},
    {'eps': 0.4, 'min_samples': 2},
    {'eps': 0.5, 'min_samples': 3},
]
eval_algorithm('DBSCAN', DBSCAN, dbscan_params, None, None)


ap_params = [
    {'damping': 0.5, 'random_state': 0},
    {'damping': 0.7, 'random_state': 0},
    {'damping': 0.9, 'random_state': 0},
    {'damping': 0.95, 'random_state': 0},
    {'damping': 0.99, 'random_state': 0},
]
eval_algorithm('AffinityPropagation', AffinityPropagation, ap_params, None, None)


agg_params = [
    {'n_clusters': 2, 'linkage': 'ward'},
    {'n_clusters': 3, 'linkage': 'ward'},
    {'n_clusters': 4, 'linkage': 'ward'},
    {'n_clusters': 2, 'linkage': 'complete'},
    {'n_clusters': 3, 'linkage': 'average'},
]
eval_algorithm('Agglomerative', AgglomerativeClustering, agg_params, None, None)

spectral_params = [
    {'n_clusters': 2, 'affinity': 'nearest_neighbors', 'n_neighbors': 5},
    {'n_clusters': 3, 'affinity': 'nearest_neighbors', 'n_neighbors': 5},
    {'n_clusters': 4, 'affinity': 'nearest_neighbors', 'n_neighbors': 5},
    {'n_clusters': 2, 'affinity': 'nearest_neighbors', 'n_neighbors': 10},
    {'n_clusters': 3, 'affinity': 'nearest_neighbors', 'n_neighbors': 10},
]
eval_algorithm('Spectral', SpectralClustering, spectral_params, None, None)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


KMeans
  params={'n_clusters': 2, 'n_init': 10, 'random_state': 0}
    средний ARI: 0.0225  суммарное время: 0.09 c
  params={'n_clusters': 3, 'n_init': 10, 'random_state': 0}
    средний ARI: 0.0942  суммарное время: 0.13 c
  params={'n_clusters': 4, 'n_init': 10, 'random_state': 0}
    средний ARI: 0.0949  суммарное время: 0.12 c
  params={'n_clusters': 5, 'n_init': 10, 'random_state': 0}
    средний ARI: 0.0844  суммарное время: 0.15 c
  params={'n_clusters': 6, 'n_init': 10, 'random_state': 0}
    средний ARI: 0.0635  суммарное время: 0.18 c

DBSCAN
  params={'eps': 0.1, 'min_samples': 1}
    средний ARI: 0.0011  суммарное время: 0.02 c
  params={'eps': 0.2, 'min_samples': 1}
    средний ARI: 0.0044  суммарное время: 0.02 c
  params={'eps': 0.3, 'min_samples': 2}
    средний ARI: -0.0292  суммарное время: 0.02 c
  params={'eps': 0.4, 'min_samples': 2}
    средний ARI: -0.0101  суммарное время: 0.01 c
  params={'eps': 0.5, 'min_samples': 3}
    средний ARI: 0.0198  суммарное время:

/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(


  params={'damping': 0.99, 'random_state': 0}
    средний ARI: 0.0218  суммарное время: 0.15 c

Agglomerative
  params={'n_clusters': 2, 'linkage': 'ward'}
    средний ARI: -0.0120  суммарное время: 0.10 c
  params={'n_clusters': 3, 'linkage': 'ward'}
    средний ARI: 0.0347  суммарное время: 0.02 c
  params={'n_clusters': 4, 'linkage': 'ward'}
    средний ARI: 0.0386  суммарное время: 0.02 c
  params={'n_clusters': 2, 'linkage': 'complete'}
    средний ARI: 0.0815  суммарное время: 0.02 c
  params={'n_clusters': 3, 'linkage': 'average'}
    средний ARI: -0.0092  суммарное время: 0.02 c

Spectral
  params={'n_clusters': 2, 'affinity': 'nearest_neighbors', 'n_neighbors': 5}
    средний ARI: -0.0172  суммарное время: 0.08 c
  params={'n_clusters': 3, 'affinity': 'nearest_neighbors', 'n_neighbors': 5}
    средний ARI: 0.1282  суммарное время: 0.08 c
  params={'n_clusters': 4, 'affinity': 'nearest_neighbors', 'n_neighbors': 5}
    средний ARI: 0.1797  суммарное время: 0.09 c
  params={'n_c

Получается, birch сначала строит дерево из подкластеров и из них собирает финальные кластеры. Видимо, чтобы вычисления проиъодили по всем точкам сразу. На аутпуте листья дерева дополнительно прогоняются через кластеризацию, чтобы получить нужное число групп